# Mastering Tools: The Agent's Gateway to the World

So far, our agents could reason, loop, and update memory.
However, they were still limited to **conversation only**.

Real-world agents must do more than talk.
They must:

- retrieve information,
- perform precise calculations,
- interact with external systems,
- trigger real actions.

This is where **tools** enter the picture.

Tools allow an agent to extend its capabilities beyond language generation.
They connect reasoning to action.

In [33]:
import os
import json
from dotenv import load_dotenv
from typing import TypedDict, Annotated, List

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers.json import JsonOutputParser
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, ToolMessage
from langchain_openai import ChatOpenAI
from langchain_anthropic import ChatAnthropic
from langgraph.graph import StateGraph, END

load_dotenv()

GPT_MODEL = "gpt-5-nano"
ANTHROPIC_MODEL = "claude-opus-4-6"

## The Role of Tools in Agentic Reasoning

Why do agents need tools at all?

Because large language models, even powerful ones, have structural limitations.

### 1) Knowledge Cutoff

LLMs are trained on data up to a certain date.
They do not have access to live information.

Example:
- Asking for today’s stock price,
- Requesting the current weather,
- Checking breaking news.

Without a search tool or API call, the model must guess.

A **search tool** or a live API call solves this limitation.

### 2) Inability to Act

An LLM can describe how to send an email,
but it cannot actually send one.

It can explain how to book a ticket,
but it cannot complete the transaction.

To interact with the world,
the agent must call real code:
- an email API,
- a payment gateway,
- a database,
- a calendar service.

Tools give the agent the ability to act.

### 3) Lack of Determinism and Precision

LLMs are probabilistic systems.

They may:
- approximate math,
- hallucinate structured data,
- produce slightly inconsistent outputs.

A **calculator tool** provides perfect mathematical precision.
A **database query** returns exact values.
A **validation function** ensures structure correctness.

Tools provide reliability where the model cannot.


## Tools as an Extension of Intelligence

An agent’s intelligence is not only in language generation.

It lies in its ability to recognize when:

- its internal knowledge is insufficient,
- precision is required,
- or action must be taken.

This reasoning process is often described by the **ReAct framework**:

Reason → Act → Observe → Repeat

### ReAct in Practice

- **Thought (Reason)**  
  The LLM analyzes the user request.  
  It decides whether it can answer directly or needs a tool.  
  If a tool is needed, it outputs a structured request (e.g., JSON).

- **Action**  
  The agent’s code (our LangGraph graph) intercepts this structured request  
  and executes the corresponding Python function.

- **Observation**  
  The result from the tool is captured and stored.

- **Loop**  
  The observation is sent back to the LLM,  
  which decides whether to answer or use another tool.

This loop continues until a final response is produced.

## Illustrating the “Thought” Step

We will not build the full loop yet.

Instead, we focus on one crucial part:
teaching the LLM to decide when to use a tool
and to format its decision in a structured way.

We start by defining a simple weather tool.

In [3]:
# 1. Define the Tool as a Python function

def get_weather(city: str) -> str:
    """A simple tool that returns a hardcoded weather for a given city."""
    
    if "lome" in city.lower():
        return f"The weather in {city} is currently 29°C and sunny."
    elif "renne" in city.lower():
        return f"The weather in {city} is currently 15°C and cloudy."
    else:
        return "Sorry, I don't have the weather for that city."


Now we instruct the LLM how to use this tool.

The key idea:
If a tool is needed, the model must respond with a JSON object
containing:
- `tool_name`
- `tool_input`

In [15]:
# 2. Instruct the LLM how to use our tool

prompt = ChatPromptTemplate.from_messages(
    [
        ("system",
         """
You are a helpful assistant. You have access to a single tool: get_weather.

If you need the tool, respond ONLY with valid JSON in this exact format:
{{"tool_name": "get_weather", "tool_input": "<city>"}}

If you do not need a tool, respond with normal text (not JSON).
         """.strip()
        ),
        ("human", "{user_input}")
    ]
)

We now bind the model to enforce structured JSON output.

In [16]:
# 3. Create the Chain 

model = ChatAnthropic(model=ANTHROPIC_MODEL, temperature=0)

chain = prompt | model

If the question requires live information,
the model should generate a structured tool request.

In [17]:
# 4. Invoke the Chain with a relevant question

tool_request = chain.invoke({"user_input": "What is the weather in Lome?"})

print(f"Model Output (String):\n{tool_request.content}")

parsed_output = json.loads(tool_request.content)
print(f"\nModel Output (dictionary):\n{parsed_output}")

if parsed_output.get("tool_name") == "get_weather":
    city = parsed_output.get("tool_input")
    observation = get_weather(city)
    print(f"\nObservation from tool:\n{observation}")

Model Output (String):
{"tool_name": "get_weather", "tool_input": "Lome"}

Model Output (dictionary):
{'tool_name': 'get_weather', 'tool_input': 'Lome'}

Observation from tool:
The weather in Lome is currently 29°C and sunny.


If the question does not require a tool,
the model should answer directly.

In [18]:
# 5. Invoke the chain with a question that does not require a tool

chain = prompt | model
direct_answer = chain.invoke({"user_input": "Hello, how are you today?"})

print(f"Model's direct answer:\n{direct_answer.content}")

Model's direct answer:
Hello! I'm doing great, thank you for asking! 😊 How are you doing today? Is there anything I can help you with?


> What We Just Demonstrated ?

We implemented the first step of a tool-using agent:

- The LLM reasons about whether a tool is needed.
- It produces a structured request.
- The agent executes the tool.
- The result becomes an observation.

In the next section, we will move beyond custom functions
and explore **pre-built tools** such as:

- web search,
- file system access,
- and shell execution.

These tools connect agents to real environments.

## Using Pre-built Tools: Search, File System, and Shell Access

Building custom tools gives you full control.
However, many agent tasks are common:

- searching the web,
- reading local files,
- listing directories,
- executing system commands.

Rewriting these capabilities from scratch every time would be inefficient.

For this reason, the LangChain ecosystem provides a large collection of **pre-built tools**.
These tools are ready to use and already formatted in a way that LLMs can understand.

They allow you to connect agents to real-world environments with minimal effort.

### Accessing the Internet: The Tavily Search Tool

To overcome an LLM’s knowledge cutoff, a web search tool is essential.

While several search providers exist, `TavilySearchResults` is particularly well-suited for agents because:

- its API is optimized for AI use,
- it returns concise, structured results,
- the output is easier for models to process.

First, obtain a free API key from the Tavily website.

In [ ]:
# .env file
TAVILY_API_KEY="tvly-..."

Next you will need to install the Tavily integration package:

In [ ]:
pip install tavily-python

Now, let's see how easy it is to instantiate and use this tool directely.

In [29]:
from langchain_community.tools.tavily_search import TavilySearchResults

load_dotenv()


seach_tool = TavilySearchResults(max_results=2)

query = "What is the current status of the Artemis program?"

result = seach_tool.invoke(query)

print(f"---- Tavily Search Results for : '{query}' -----")
print(result)

---- Tavily Search Results for : 'What is the current status of the Artemis program?' -----
[{'title': 'Artemis - NASA', 'url': 'https://www.nasa.gov/humans-in-space/artemis/', 'content': 'The Extravehicular Activity and Human Surface Mobility Program serves as NASA’s program to develop next-generation spacesuits, human-rated rovers, tools, and the spacewalk support systems that will enable astronauts to survive and work outside the confines of a spacecraft to explore on and around the Moon.\n\nArtist concept of Artemis astronaut working on Lunar surface.\n\n## Landing on the Moon\n\nWorking with American companies to deliver science and technology to the Moon’s surface. [...] NASA is working with industry to develop the human landing systems, or next-generation landers, that will safely carry Artemis astronauts from lunar orbit to the\u2002Moon’s surface and back for future crewed exploration of the Moon. The agency will use SpaceX’s Starship Human Landing System for Artemis III and A

At this stage, the tool works independently.
The next step is to make the LLM aware of it.

### Making the Model Tool-Aware with `.bind_tools()`

How does the model know that a tool exists?

The modern method is `.bind_tools()`.

When you call:

```python
model.bind_tools([tool1, tool2])
```

LangChain:

- Inspects each tool (its function signature and docstring).

- Converts it into a structured schema.

- Automatically injects that schema into every model call.

This allows the LLM to reason about available tools.
If it decides a tool is needed, it does not return plain text.
Instead, it returns an AIMessage containing a tool_calls field.

In [24]:
tools = [TavilySearchResults(max_results=2)]

model = ChatAnthropic(model=ANTHROPIC_MODEL, temperature=0)
model_with_tools = model.bind_tools(tools)

query = "What is the current status of the Artemis program?"

ai_response = model_with_tools.invoke(query)

print("---- AI Response -----")
print(f"Content: {ai_response.content}")
print(f"Tool calls: {ai_response.tool_calls}")

if ai_response.tool_calls:
    
    first_tool_call = ai_response.tool_calls[0]
    tool_name = first_tool_call["name"]
    tool_args = first_tool_call["args"]
    
    print(f"\nModel wants to call the '{tool_name}' tool with the following arguments: {tool_args}")

---- AI Response -----
Content: [{'id': 'toolu_01GkjiKTU514tfVaT7vzvfTi', 'input': {'query': 'Artemis program current status 2024 2025'}, 'name': 'tavily_search_results_json', 'type': 'tool_use', 'caller': {'type': 'direct'}}]
Tool calls: [{'name': 'tavily_search_results_json', 'args': {'query': 'Artemis program current status 2024 2025'}, 'id': 'toolu_01GkjiKTU514tfVaT7vzvfTi', 'type': 'tool_call'}]

Model wants to call the 'tavily_search_results_json' tool with the following arguments: {'query': 'Artemis program current status 2024 2025'}


If `tool_calls` is not empty, the model has decided that using a tool is the best way to answer the question.

This is the practical implementation of the **Reason → Act** transition.

### Using Multiple Tools: Search and Shell

An agent becomes more powerful as you give it more tools.

You can pass multiple tools to `.bind_tools()`,
and the model will choose the most appropriate one based on the user request.

We will combine:

- a search tool (Tavily),
- a shell tool (system-level access).

⚠️ **Important Warning**

The `ShellTool` is extremely powerful and potentially dangerous.
It can execute arbitrary commands on your machine.

Never expose an agent with `ShellTool` to:
- public internet access,
- untrusted users,
- production environments without sandboxing.

Use it only in controlled environments.

In [26]:
from langchain_community.tools import ShellTool

search_tool = TavilySearchResults(max_results=2)
shell_tool = ShellTool()

tools = [search_tool, shell_tool]

model = ChatAnthropic(model=ANTHROPIC_MODEL, temperature=0)
model_with_tools = model.bind_tools(tools)

> Scenario A: The model should choose the search tool

In [27]:
query_a = "What is the current price of Bitcoin?"

ai_response_a = model_with_tools.invoke(query_a)

print("---- Scenario A: Search Tool -----")
print(f"Content: {ai_response_a.content}")
print(f"Tool calls: {ai_response_a.tool_calls}")

---- Scenario A: Search Tool -----
Content: [{'id': 'toolu_01KcAyJuUhMxFcGBxjaEd3ZV', 'input': {'query': 'current price of Bitcoin'}, 'name': 'tavily_search_results_json', 'type': 'tool_use', 'caller': {'type': 'direct'}}]
Tool calls: [{'name': 'tavily_search_results_json', 'args': {'query': 'current price of Bitcoin'}, 'id': 'toolu_01KcAyJuUhMxFcGBxjaEd3ZV', 'type': 'tool_call'}]


If the model reasons correctly,
it will generate a tool call targeting the search tool.

> Scenario B: The model should choose the shell tool

In [28]:
query_b = "List the files in my current directory, with details."

ai_response_b = model_with_tools.invoke(query_b)

print("\n---- Scenario B: Shell Tool -----")
print(f"Content: {ai_response_b.content}")
print(f"Tool calls: {ai_response_b.tool_calls}")


---- Scenario B: Shell Tool -----
Content: [{'id': 'toolu_012o833vqvkTsVV72zEemZZq', 'input': {'commands': 'ls -la'}, 'name': 'terminal', 'type': 'tool_use', 'caller': {'type': 'direct'}}]
Tool calls: [{'name': 'terminal', 'args': {'commands': 'ls -la'}, 'id': 'toolu_012o833vqvkTsVV72zEemZZq', 'type': 'tool_call'}]


In this case, the model recognizes that the request requires system access,
and it selects the `ShellTool` and even provide the appropriate command `ls -la` as args to `ShellTool`.

This demonstrates how tool selection becomes part of the agent’s reasoning process.

The LLM is no longer just generating answers.
It is generating **structured decisions** about actions to perform.

## Creating Custom Tools from Python Functions with Pydantic for Type Hinting

Pre-built tools are convenient, but real agentic systems rarely rely only on generic capabilities.

In production, your agent will need to interact with:

- your own internal APIs,
- your business logic,
- your database layer,
- your company-specific workflows.

This is where custom tools become essential.

Custom tools allow you to expose **your own Python functions** to the LLM,
so it can reason about them and decide when to use them.

> The `@tool` Decorator

The modern and most straightforward way to create a tool in LangChain
is to use the `@tool` decorator.

You simply:

1. Write a normal Python function.
2. Add `@tool` above it.

The decorator automatically:

- inspects the function name,
- reads the docstring,
- extracts argument names and type hints,
- converts everything into a structured schema,
- makes it compatible with `.bind_tools()`.

This schema is what the LLM sees when deciding which tool to call.

For the LLM to use your tool effectively, three elements are critical:

> 1) **Function Name**

The function name becomes the **tool name**.

It must be:
- clear,
- descriptive,
- unambiguous.

Bad example:
`do_it()`

Good example:
`multiply()`
`send_email()`
`book_flight()`

> 2) **Docstring (Most Important)**

The docstring becomes the tool’s **description**.

The LLM relies heavily on this description to decide:

- when the tool is appropriate,
- what the tool does,
- what kind of problem it solves.

If the description is vague, the agent will reason poorly.

Think of the docstring as the tool’s “instruction manual” for the LLM.

> 3) **Function Arguments and Type Hints**

Type hints define the expected input format.

They tell the LLM:

- what arguments are required,
- what type each argument must be,
- how to structure the tool call.

If you use:

```python
def multiply(a: int, b: int) -> int:
```

The LLM knows: it needs two integers, named a and b. This improves structured tool calling dramatically.

### Creating a Simple Custom Tool

Let's start by creating a simple `multiply` tool. This is a task LLMs can often fail at, making it a perfect candidate for a deterministic, precise tool.

In [31]:
from langchain_core.tools import tool

@tool
def multiply(a: int, b: int) -> int:
    """
    Docstring for multiply

    A tool to multiply two integers. Use this for any mathematical multiplication.
    
    For example, to find the product of 5 and 20 use this tool.
    """
    print(f"---TOOL `multiply` called with a={a} and b={b}---")
    
    return a * b

tools = [multiply]

model = ChatAnthropic(model=ANTHROPIC_MODEL, temperature=0)
model_with_tools = model.bind_tools(tools)

query = "What is 37 multiplied by 55?"

ai_response = model_with_tools.invoke(query)

print("---- AI Response -----")
print(f"Content: {ai_response.content}")
print(f"Tool calls: {ai_response.tool_calls}")

---- AI Response -----
Content: [{'id': 'toolu_01JTQBvmU4YwMpzxWpfJoe6i', 'input': {'a': 37, 'b': 55}, 'name': 'multiply', 'type': 'tool_use', 'caller': {'type': 'direct'}}]
Tool calls: [{'name': 'multiply', 'args': {'a': 37, 'b': 55}, 'id': 'toolu_01JTQBvmU4YwMpzxWpfJoe6i', 'type': 'tool_call'}]


When you run this, you'll see that the model contains a `tool_calls`nattribut intructing our agent to call the `multiply` tool with the arguments `{'a':37, 'b':55}`. 

The LLM understood from the docstring that multiplication requires a tool. It extracted arguments correctly and structured them based on the type hints.

This is deterministic reasoning combined with structured action.

### Using Pydantic for Complex Structured Arguments

Simple arguments work well for basic tools.

But real-world tools often require structured inputs.

Example:
Booking a flight requires:

- passenger name,
- departure city,
- destination city,
- departure date,
- preferred airline.

Passing all of these as separate function arguments can become messy.

A cleaner solution is to use **Pydantic models**.

Pydantic allows you to:

- define structured schemas,
- validate input automatically,
- attach descriptions to each field,
- provide strict type checking.

The LLM receives this schema and generates properly structured tool calls.

In [32]:
from pydantic.v1 import BaseModel, Field

class BookFlightArgs(BaseModel):
    passenger_name: str = Field(description="The name of the passenger.")
    departure_city: str = Field(description="The city from which the flight departs.")
    destination_city: str = Field(description="The city to which the flight is headed.")
    departure_date: str = Field(description="The date of departure in YYYY-MM-DD format.")
    preferred_airline: str = Field(description="The passenger's preferred airline, if any.")

@tool  
def book_flight(args: BookFlightArgs) -> str:
    """
    Books a flight for a passenger with the specified details.
    """
    print(f"---TOOL `book_flight` called with args={args}---")
    
    return f"Flight successfully booked for {args.passenger_name} from {args.departure_city} to {args.destination_city} on {args.departure_date} with {args.preferred_airline}."

tools = [book_flight]

model = ChatAnthropic(model=ANTHROPIC_MODEL, temperature=0)

model_with_tools = model.bind_tools(tools)

query = "Please book a flight for Reginald Smith. He wants to fly from Lome to Rennes on December 15, 2026. He prefers to fly with Aire France."

response = model_with_tools.invoke(query)

print("--- Model Responsee ---")

print(f"Content: {response.content}")
print(f"Tool calls: {response.tool_calls}")

--- Model Responsee ---
Content: [{'text': "\n\nI'll book that flight for Reginald Smith right away!", 'type': 'text'}, {'id': 'toolu_01Fkefhw4BYhSk47NH6HpJgx', 'input': {'v__args': ['Reginald Smith', 'Lome', 'Rennes', '2026-12-15', 'Aire France']}, 'name': 'book_flight', 'type': 'tool_use', 'caller': {'type': 'direct'}}]
Tool calls: [{'name': 'book_flight', 'args': {'v__args': ['Reginald Smith', 'Lome', 'Rennes', '2026-12-15', 'Aire France']}, 'id': 'toolu_01Fkefhw4BYhSk47NH6HpJgx', 'type': 'tool_call'}]


Custom tools transform your agent from a general assistant
into a domain-specific system.

By combining:

- clear function names,
- descriptive docstrings,
- precise type hints,
- and Pydantic validation,

you create tools that the LLM can reason about confidently and safely.

In the next step, we will integrate these custom tools directly
into a full LangGraph loop,
allowing the agent to reason, act, observe, and continue autonomously.

## Handling Tool Errors and Invalid Inputs Gracefully

In the ideal examples so far, tools always worked perfectly.
Real systems are rarely that clean.

In production, tools can fail for many reasons:
- missing data (user not found),
- invalid inputs (wrong format),
- API timeouts,
- permission errors,
- rate limits.

A production-ready agent must be **robust**.

It should not crash when a tool fails.
Instead, it should:
- capture the error,
- treat it as an observation,
- and use the observation to recover or respond clearly.

A simple and effective pattern is:

- wrap tool execution in `try...except`,
- if the tool fails, capture the error message,
- send the result (success or error) back to the LLM as a `ToolMessage`,
- let the model decide the best next answer.

This mirrors real-world systems where:
an API failure becomes a structured response that downstream logic can handle.

### Simulating a Tool That Can Fail

To demonstrate error handling, we will build a tool called `get_user_email`.

It only knows emails for a small set of users.
If the user is not found, it raises a `ValueError`.

We then simulate a single ReAct-style turn:

1. The model decides whether it needs the tool.
2. Our code executes the tool inside `try...except`.
3. The result or error becomes the observation.
4. We send this observation back using a `ToolMessage`.
5. The model uses the observation to produce the final answer.

In [35]:
# A tiny "database" of users
_USERS = {
    "yatoute": "yatoute@ai.com",
    "richard": "richard@ai.com",
    "alex": "alex@ai.com",
}

@tool
def get_user_email(username: str) -> str:
    """
    Return the email address for a given username.

    Raises:
        ValueError: if the username is not found.
    """
    username_lower = username.lower()

    if username_lower in _USERS:
        return _USERS[username_lower]

    raise ValueError(f"Unknown user: {username}")

We bind the tool to the model so it can generate structured `tool_calls`.

In [36]:
tools = [get_user_email]

model = ChatOpenAI(model=GPT_MODEL, temperature=0)
model_with_tools = model.bind_tools(tools)

### Simulating One Agent Turn (Reason → Act → Observe → Answer)

This function simulates a single agent turn:

- It sends the user query to the tool-aware model.
- If the model requests a tool call, we execute it safely.
- We append a `ToolMessage` containing either:
  - the tool result, or
  - an error message.
- We call the model again to generate the final answer based on the observation.

In [39]:
def run_agent_turn(user_query: str) -> str:
    print(f"\n==== Running agent for query: '{user_query}' ====")

    messages = [HumanMessage(content=user_query)]

    print("\n--- Reason: model decides whether to use a tool ---")
    ai_response = model_with_tools.invoke(messages)
    messages.append(ai_response)
    
    print(f"Tool calls: {ai_response.tool_calls}")
    
    # If no tool call, the model answers directly
    if not ai_response.tool_calls:
        print("\n--- No tool needed. End of turn. ---")
        return ai_response.content

    print("\n--- Act + Observe: execute tool calls with error handling ---")

    for tool_call in ai_response.tool_calls:
        tool_name = tool_call["name"]
        tool_args = tool_call["args"]

        if tool_name == "get_user_email":
            try:
                observation = get_user_email.invoke(tool_args)
                print(f"Tool succeeded. Observation: {observation}")
            except Exception as e:
                observation = f"Tool error: {type(e).__name__}: {e}"
                print(f"Tool failed. Observation: {observation}")

            messages.append(
                ToolMessage(
                    content=str(observation),
                    tool_call_id=tool_call["id"],
                )
            )

    print("\n--- Final answer: model responds using the observation ---")
    final_response = model_with_tools.invoke(messages)
    return final_response.content

### Run the Simulation (Success and Failure)

We test two real-world scenarios:

- success: user exists in the system
- failure: user is unknown, tool raises an error

In [40]:
# Scenario A: success
success_response = run_agent_turn("What is Yatoute's email address?")
print(f"\nFinal Agent Response (Success):\n{success_response}")


==== Running agent for query: 'What is Yatoute's email address?' ====

--- Reason: model decides whether to use a tool ---
Tool calls: [{'name': 'get_user_email', 'args': {'username': 'Yatoute'}, 'id': 'call_J0iWyzEQj0BxcW9ISv2Q5DIW', 'type': 'tool_call'}]

--- Act + Observe: execute tool calls with error handling ---
Tool succeeded. Observation: yatoute@ai.com

--- Final answer: model responds using the observation ---

Final Agent Response (Success):
Yatoute's email address is yatoute@ai.com.


In [41]:
# Scenario B: failure
failure_response = run_agent_turn("What is Samuel's email address?")
print(f"\nFinal Agent Response (Failure):\n{failure_response}")


==== Running agent for query: 'What is Samuel's email address?' ====

--- Reason: model decides whether to use a tool ---
Tool calls: [{'name': 'get_user_email', 'args': {'username': 'Samuel'}, 'id': 'call_kxJOMKoFEECqoAVhNc6F3crB', 'type': 'tool_call'}]

--- Act + Observe: execute tool calls with error handling ---
Tool failed. Observation: Tool error: ValueError: Unknown user: Samuel

--- Final answer: model responds using the observation ---

Final Agent Response (Failure):
I can’t find a user with the username "Samuel" in our directory. It might be a different username or a different Samuel.

Could you provide:
- the exact username (if you know it), or
- the full name (e.g., Samuel Johnson) plus any known department or team?

With a bit more detail, I can try again.


> Analyzing the Results

In the success case:
- the tool returns a valid email,
- the LLM uses it to answer directly.

In the failure case:
- the tool returns an error observation,
- the LLM can explain the problem instead of crashing.

This pattern scales to real tool failures such as:
- API timeouts,
- database connection errors,
- validation failures,
- missing permissions.

The main idea remains the same:
**errors become observations**, and observations drive the next reasoning step.

> Key Takeaway

Tool errors are not special cases to ignore.
They are part of the agent’s environment.

A robust agent does not break when tools fail.
It captures failures, learns from them, and responds intelligently.

Next, we will provide multiple tools and let the agent choose the best one for each request.

## Providing Multiple Tools and Letting the Agent Choose

An agent with a single tool is useful, but limited.
It can only solve one type of problem reliably.

The real power of an agent appears when it has a **toolkit**:
several tools that cover different needs, and the ability to choose the right one.

This decision-making is not magic.

When you bind a list of tools to a model using `.bind_tools(tools)`,
the model receives a description of every tool:
- the tool name,
- the docstring (purpose + when to use it),
- and the argument schema (from type hints / Pydantic).

The model then uses reasoning to match the user’s intent to the best tool.

This is why tool quality matters:
a vague docstring produces vague decisions.
A precise docstring produces reliable tool selection.

### Building a Multi-Tool Agent

We will build an agent equipped with two custom tools:

- `get_current_time`: for current time and date in a given timezone  
- `multiply`: for precise multiplication

Then we will test the agent with different prompts to see whether it:
- selects one tool,
- selects the other tool,
- selects both,
- or decides that no tool is needed.

In [45]:
from datetime import datetime
import pytz

@tool
def get_current_time(timezone: str) -> str:
    """
    Return the current date and time in the format YYYY-MM-DD HH:MM:SS
    for a given timezone.

    Use this tool for questions about the current time or current date
    in a specific location.

    Example: "What time is it in Africa/Lome?"
    """
    print(f"--- TOOL `get_current_time` called with timezone='{timezone}' ---")
    return datetime.now(pytz.timezone(timezone)).strftime("%Y-%m-%d %H:%M:%S")

@tool
def multiply(a: float, b: float) -> float:
    """
    Multiply two numbers.

    Use this tool whenever multiplication is needed to answer a question.

    Example: "What is 5 multiplied by 20?"
    """
    print(f"--- TOOL `multiply` called with a={a} and b={b} ---")
    return a * b

Now bind both tools to the model.  
The model can choose either one depending on the user query.

In [46]:
tools = [get_current_time, multiply]

model = ChatOpenAI(model=GPT_MODEL, temperature=0)
model_with_tools = model.bind_tools(tools)

> A Helper Function to Run One Agent Turn

This function implements a one-turn ReAct cycle:

1) send the user query  
2) the model optionally produces `tool_calls`  
3) we execute the requested tools  
4) we send observations back using `ToolMessage`  
5) the model generates the final answer

In [ ]:
def run_agent_turn(query: str) -> str:
    messages = [HumanMessage(content=query)]

    print(f"\n==== Model called with query: {query!r} ====")
    ai_response = model_with_tools.invoke(messages)

    # If no tools are requested, return the direct answer
    if not ai_response.tool_calls:
        print("\n--- Model answers directly ---")
        return ai_response.content
    
    print(f"\nTool calls: {ai_response.tool_calls}")
    messages.append(ai_response)

    print("\n--- Executing tools ---")

    for tool_call in ai_response.tool_calls:
        tool_name = tool_call["name"]
        tool_args = tool_call["args"]

        print(f"\nCalling tool: {tool_name} with args: {tool_args}")

        try:
            if tool_name == "get_current_time":
                observation = get_current_time.invoke(tool_args)

            elif tool_name == "multiply":
                observation = multiply.invoke(tool_args)

            else:
                observation = f"Tool '{tool_name}' is not available."

        except Exception as e:
            observation = f"Tool error: {type(e).__name__}: {e}"

        print(f"Observation: {observation}")

        messages.append(
            ToolMessage(
                content=str(observation),
                tool_call_id=tool_call["id"],
            )
        )

    print("\n--- Final answer: model responds using the observation ---")
    final_response = model_with_tools.invoke(messages)
    return final_response.content

> Scenario A: The agent should use the time tool

Real-world analogy:
a chatbot in a global company answering time questions for different offices.

In [48]:
print("\n--- Scenario A ---")
print(run_agent_turn("What time is it right now in Togo/Lome?"))


--- Scenario A ---

==== Model called with query: 'What time is it right now in Togo/Lome?' ====

Tool calls: [{'name': 'get_current_time', 'args': {'timezone': 'Africa/Lome'}, 'id': 'call_aLfG17edfug8mFM5ru77MpZe', 'type': 'tool_call'}]

--- Executing tools ---

Calling tool: get_current_time with args: {'timezone': 'Africa/Lome'}
--- TOOL `get_current_time` called with timezone='Africa/Lome' ---
Observation: 2026-02-21 12:29:25

--- Final answer: model responds using the observation ---
It's 12:29:25 PM on February 21, 2026 in Lomé, Togo (Africa/Lome, GMT+0). Would you like this converted to another time zone or set as a reminder?


> Scenario B: The agent should use the multiplication tool

Real-world analogy:
calculating areas, prices, totals, or quantities where precision matters.

In [52]:
print("\n--- Scenario B ---")
print(run_agent_turn("Find the area of a rectangular field with length 200 m and width 150 m. Please use tools provided."))


--- Scenario B ---

==== Model called with query: 'Find the area of a rectangular field with length 200 m and width 150 m. Please use tools provided.' ====

Tool calls: [{'name': 'multiply', 'args': {'a': 200, 'b': 150}, 'id': 'call_IjxfveRO3X0ej4s1bD8YSdVH', 'type': 'tool_call'}]

--- Executing tools ---

Calling tool: multiply with args: {'a': 200, 'b': 150}
--- TOOL `multiply` called with a=200.0 and b=150.0 ---
Observation: 30000.0

--- Final answer: model responds using the observation ---
The area of a rectangle is length times width.
Area = 200 m × 150 m = 30,000 square meters (or 3 hectares).


> Scenario C: The agent should use both tools

Here we force a query that naturally requires:
- the current time (time tool)
- and a multiplication (multiply tool)

This is a realistic pattern for assistants that combine:
live information + computation.

In [53]:
print("\n--- Scenario C ---")
print(run_agent_turn(
    "In Paris, what time is it right now, and what is 37 multiplied by 55?"
))


--- Scenario C ---

==== Model called with query: 'In Paris, what time is it right now, and what is 37 multiplied by 55?' ====

Tool calls: [{'name': 'get_current_time', 'args': {'timezone': 'Europe/Paris'}, 'id': 'call_Er8fRSpA04z3ggQyhuPhkRYP', 'type': 'tool_call'}, {'name': 'multiply', 'args': {'a': 37, 'b': 55}, 'id': 'call_LPgPO8SujgIJQshlF5sa5hgh', 'type': 'tool_call'}]

--- Executing tools ---

Calling tool: get_current_time with args: {'timezone': 'Europe/Paris'}
--- TOOL `get_current_time` called with timezone='Europe/Paris' ---
Observation: 2026-02-21 13:36:50

Calling tool: multiply with args: {'a': 37, 'b': 55}
--- TOOL `multiply` called with a=37.0 and b=55.0 ---
Observation: 2035.0

--- Final answer: model responds using the observation ---
- Paris local time: 2026-02-21 13:36:50
- 37 multiplied by 55: 2035


> Key Takeaway

Giving an agent multiple tools is not enough.
The agent also needs:
- good tool descriptions,
- clear schemas,
- and a reliable execution loop.

Once those pieces are in place, tool choice becomes a natural part of reasoning.

## Sequential Tool Use for Multi-Step Problems

Choosing the right tool is already useful.
But advanced agents can go further:

They can solve complex tasks by:
- breaking a problem into smaller steps,
- calling multiple tools in sequence,
- using each observation to decide the next action.

This is the core strength of the ReAct loop:
**Reason → Act → Observe → repeat**.

A realistic example looks like this:

- **Action 1:** search for the Best Picture winner  
- **Observation 1:** obtain the movie title  
- **Action 2:** search for a person related to that movie (actor/director)  
- **Observation 2:** obtain the person’s name  
- **Action 3:** search for a fact about the person (birth year, birthplace, etc.)  
- **Observation 3:** obtain the missing fact  
- **Action 4:** run a calculation (if needed)  
- **Observation 4:** obtain the exact number  
- **Final Answer:** combine everything into one response

The key point is that the agent does not need to know the whole plan in advance.
It can decide step by step, based on the latest observation.

> **A Multi-Turn Simulation with a Search Tool**

We now simulate this behavior with a loop.

The agent will:
1) receive the user query,
2) decide to call the search tool,
3) read the tool result (observation),
4) decide whether another search is needed,
5) stop when it has enough information to answer.

This is similar to real workflows such as:
- investigating a topic with multiple searches,
- collecting facts from different sources,
- iterating until the response is complete.

We create the tool and bind it to the model.  
Once bound, the model can decide when to call it.

In [62]:
search_tool = TavilySearchResults()
tools = [search_tool]
model = ChatOpenAI(model=GPT_MODEL, temperature=0).bind_tools(tools)

Now we will build a Multi-Step Agent Loop. It runs turns until the model stops requesting tools.

Important details:
- After each assistant message with `tool_calls`, we append it to `messages`.
- After each tool execution, we send back a `ToolMessage` linked by `tool_call_id`.
- When no tool call is produced, the model is ready to answer.

In [66]:
def run_multi_step_agent(query: str, max_turns: int = 6) -> str:
    messages = [HumanMessage(content=query)]

    print(f"\n--- Starting multi-step query: {query!r} ---")

    for turn in range(1, max_turns + 1):
        print(f"\n--- Turn {turn} ---")

        ai_response = model.invoke(messages)
        print(f"Tool calls: {ai_response.tool_calls}")

        # If the model is done, return the final answer
        if not ai_response.tool_calls:
            print("\n--- No more tool calls. Final answer produced. ---")
            return ai_response.content

        # Keep the assistant message (contains tool_calls)
        messages.append(ai_response)

        print("\n--- Executing tool calls ---")

        for tool_call in ai_response.tool_calls:
            tool_name = tool_call["name"]
            tool_args = tool_call["args"]

            print(f"Calling tool: {tool_name} with args: {tool_args}")

            try:
                
                if tool_name == search_tool.name:
                    observation = search_tool.invoke(tool_args)
                else:
                    observation = f"Tool '{tool_name}' not found."

            except Exception as e:
                observation = f"Tool error: {type(e).__name__}: {e}"

            print(f"Observation: {observation}")

            messages.append(
                ToolMessage(
                    content=str(observation),
                    tool_call_id=tool_call["id"],
                )
            )

    return "Stopped: max_turns reached before the agent produced a final answer."

Try It on a Query That Naturally Requires Multiple Searches

In [68]:
query = (
    "Who was the director of the movie that won Best Picture at the 77th Academy Awards, "
    "and where was he born?"
    "Please only use tool provided sequentially, many time you need."
)

final_answer = run_multi_step_agent(query)
print("\n--- FINAL ANSWER ---")
print(final_answer)


--- Starting multi-step query: 'Who was the director of the movie that won Best Picture at the 77th Academy Awards, and where was he born?Please only use tool provided sequentially, many time you need.' ---

--- Turn 1 ---
Tool calls: [{'name': 'tavily_search_results_json', 'args': {'query': '77th Academy Awards Best Picture winner'}, 'id': 'call_GN1oYQ8Em0iZH1Lst3v3LE8b', 'type': 'tool_call'}]

--- Executing tool calls ---
Calling tool: tavily_search_results_json with args: {'query': '77th Academy Awards Best Picture winner'}
Observation: [{'title': '77th Academy Awards - IMDb', 'url': 'https://www.imdb.com/list/ls538662330/', 'content': "Winner - Best Picture  \n  Winner - Best Director for Clint Eastwood  \n  Nominated - Best Actor for Clint Eastowood  \n  Winner - Best Actress for Hilary Swank  \n  Winner - Best Supporting Actor for Morgan Freeman  \n  Nominated - Best Adapted Screenplay for Paul Haggis  \n  Nominated - Best Film Editing for Joel Cox\n ### 2. The Aviator\n\n  2004

In this example, we used a single search tool to demonstrate how an agent can reason across multiple turns.

The important idea is not the specific tool, but the pattern:

- The agent gathers information step by step.
- Each observation influences the next decision.
- The loop continues until enough information is collected.
- A final answer is synthesized from all intermediate results.

This iterative reasoning process is what transforms a simple assistant into a structured problem-solver.

So far, our agent has been able to:
- call tools,
- handle errors,
- choose between multiple tools,
- and execute multi-step reasoning chains.

However, each interaction has still been limited to a single session.
Once the turn ends, the context disappears.

In the next lesson, we will address this limitation by introducing a new capability:

**Memory.**

We will explore how to persist information across turns,
how to give agents long-term context,
and how memory changes the way agents reason and behave over time.